In [ ]:
!pip install --upgrade sentence-transformers transformers

  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.1.0-py3-none-any.whl (483 kB)


In [ ]:
!pip install --upgrade bitsandbytes

In [ ]:
!pip install torch==2.3.0 torchvision==0.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99

In [ ]:
import os
import json
import requests
import time
import random
import pandas as pd
import numpy as np
import csv
import random
import gc
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import jaccard_score
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import ndcg_score, mean_squared_error
# from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

In [ ]:
import os
os.listdir()

['.config',
 'main_metadata.json',
 'similar_questions.csv',
 'drive',
 'sample_data']

# Fetching Data

In [ ]:
url="https://leetcode.com/graphql"

allQ_query="""
query problemsetQuestionList($categorySlug: String, $limit: Int, $skip: Int, $filters: QuestionListFilterInput){
    problemsetQuestionList: questionList(
        categorySlug: $categorySlug
        limit: $limit
        skip: $skip
        filters: $filters
    ){
        total: totalNum
        questions: data{
            questionId
            questionFrontendId
            title
            titleSlug
        }
    }
}
"""

question_query="""
query questionData($titleSlug: String!){
    question(titleSlug: $titleSlug){
        questionId
        questionFrontendId
        title
        content
        likes
        dislikes
        stats
        similarQuestions
        categoryTitle
        hints
        topicTags { name }
        companyTags { name }
        difficulty
        isPaidOnly
        solution { canSeeDetail content }
        hasSolution
        hasVideoSolution
    }
}
"""

In [ ]:
def fetch_all_questions():
    variables={
        "categorySlug":"",
        "limit":10000,
        "skip":0,
        "filters":{}
    }
    payload={
        "query":allQ_query,
        "variables":variables
    }
    response=requests.post(url,json=payload)
    if response.status_code!=200:
        print("Failed to fetch data")
        print(response.json())
        exit()
    data=response.json()
    questions=data['data']['problemsetQuestionList']['questions']
    print(f"Fetched {len(questions)} questions")
    return questions

In [ ]:
def fetch_question(title_slug):
    payload={
        "query":question_query,
        "variables":{
            "titleSlug":title_slug
        }
    }
    retries=8
    base_delay=2
    for tries in range(retries):
        try:
            response=requests.post(url,json=payload,timeout=15)
            delay=random.uniform(1,2)
            time.sleep(delay)
            if response.status_code==200:
                data=response.json()
                q=data['data']['question']
                q['url']=f"https://leetcode.com/problems/{title_slug}/"
                q['titleSlug']=title_slug
                stats = json.loads(q['stats'])
                q["totalAccepted"] = stats.get("totalAccepted")
                q["totalSubmission"] = stats.get("totalSubmission")
                q['totalAcceptedRaw']=stats.get("totalAcceptedRaw")
                q['totalSubmissionRaw']=stats.get('totalSubmissionRaw')
                q["acRate"] = stats.get("acRate")
                return q
            else:
                print(f"retrying {tries+1}/{retries} for {title_slug}")
        except requests.exceptions.RequestException as e:
            print(f"Request error {title_slug}:{e}")
        backoff = base_delay * (2 ** tries) + random.uniform(0, 2)
        print(f"Retrying {tries + 1}/{retries} for {title_slug} after {backoff:.1f}s")
        time.sleep(backoff)
    print(f"Failed to fetch for {title_slug}")
    return None

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Referer": "https://leetcode.com/problemset/all/"
})

questions = fetch_all_questions()
all_questions=[]
save_every=100

# dest=os.path.abspath(os.path.join(os.path.dirname(__file__),"../../data/question_metadata.json"))
dest="question_metadata.json"
already_done=set()
if os.path.exists(dest):
    with open(dest, "r", encoding="utf-8") as f:
        existing = json.load(f)
        all_questions = existing
        already_done = {q["questionFrontendId"] for q in all_questions}
    print(f"Resuming: {len(already_done)} questions already fetched")

for i,q in enumerate(questions):
    title_slug=q['titleSlug']
    if q['questionFrontendId'] in already_done:
        continue
    print(f'{i}/{len(questions)} fetching details for {title_slug}')
    detail=fetch_question(title_slug)
    if detail:
        all_questions.append(detail)
    # Save progress every 100 questions
    if (len(all_questions) % save_every == 0) or (i == len(questions) - 1):
        with open(dest, "w", encoding="utf-8") as f:
            json.dump(all_questions, f, indent=2, ensure_ascii=False)
        print(f"Saved progress ({len(all_questions)} questions)")
    # Every 100 requests, sleep longer (random 30-120 seconds)
    if (len(all_questions) % 100 == 0):
        long_pause = random.uniform(30, 120)
        print(f"Taking a longer break: {long_pause:.1f}s")
        time.sleep(long_pause)
print(f"Saved all questions metadata")

# Creating Similar question Dataset

In [ ]:
dest='main_metadata.json'

In [ ]:
with open(dest,'r',encoding='utf-8') as f:
  metadata=json.load(f)

In [ ]:
similar=[]
for q in metadata:
  similar.append({q['titleSlug']:q['similarQuestions']})
for f in similar:
  for x,y in f.items():
    f[x]=json.loads(y)

In [ ]:
len(metadata)

3624

In [ ]:
for sim in similar:
  for x,y in sim.items():
    temp=[]
    for tags in y:
      temp.append(tags['titleSlug'])
    sim[x]=temp

In [ ]:
similar

### Positive pairs

In [ ]:
raw_data=[]
prev=set()
for sim in similar:
  for x,y in sim.items():
    for q2 in y:
      canonical_pair = tuple(sorted((x, q2)))
      if canonical_pair not in prev:
        prev.add(canonical_pair)
        raw_data.append({'question1':x,'question2':q2,'score':1.0})
len(raw_data)

### Negative pairs and Saving dataset

Guessed weights - Only for filtering - prevent pair of questions that maybe similar from being marked as a dissimilar pair

In [ ]:
weights=[0.3,0.5,0.2]

Note: Dont forget to run the baseline() function in the Grid Search section before running this

* metadata & similar from leetcode
* embeddings from repo

In [ ]:
n=len(raw_data)
for _ in range(n):
  while True:
    x=random.randint(0,3182)
    y=random.randint(0,3182)
    while x==y:
      y=random.randint(0,len(similar)-1)
    str_x=list(similar[x].keys())[0]
    str_y=list(similar[y].keys())[0]
    if str_x not in name_embeds.keys() or str_y not in name_embeds.keys():
        continue
    canonical_pair=tuple(sorted((str_x,str_y)))
    if canonical_pair not in prev and baseline(weights,str_x,str_y)<0.4:
      prev.add(canonical_pair)
      raw_data.append({'question1':str_x,'question2':str_y,'score':0.0})
      break
print(len(raw_data)-n)

In [ ]:
raw_data

In [ ]:
len(raw_data)

In [ ]:
with open('similar_questions.csv','w',newline='') as f:
  fields=['question1','question2','score']
  writer=csv.DictWriter(f,fieldnames=fields)
  writer.writeheader()
  writer.writerows(raw_data)
print("saved")

Checking for duplicates

In [ ]:
df = pd.read_csv('/similar_questions.csv')
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
with open('main_metadata.json','r',encoding='utf-8') as f:
  main_metadata=json.load(f)
data={}
for meta in main_metadata:
  data[meta['titleSlug']]={
          'title':meta['title'],
          'code':meta['code'],
          'tags':meta['tags']
      }

with open('question_data.json','w',encoding='utf-8') as f:
  json.dump(data,f,indent=2,ensure_ascii=False)

In [ ]:
data['as-far-from-land-as-possible']

In [ ]:
with open('/question_data.json','r',encoding='utf-8') as f:
    n=json.load(f)
print(len(n))

# Embeddings

In [ ]:
name_embeds={}
code_embeds={}

In [ ]:
name_model=SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [ ]:
model_name = "nomic-ai/nomic-embed-code"
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

# ✅ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Load model in 8-bit with automatic device map
model = AutoModel.from_pretrained(
    model_name,
    quantization_config=bnb_config,               # compress to 8-bit\
    device_map="auto",               # split layers across available GPUs
    offload_folder="offload",
)

print("Model loaded successfully!")

In [ ]:
def get_embeddings(snippets, batch_size=2, max_length=512):
    embeddings = []
    for i in range(0, len(snippets), batch_size):
        batch = snippets[i:i+batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # mean pooling
        batch_embeds = outputs.last_hidden_state.mean(dim=1)
        embeddings.extend(batch_embeds.cpu().numpy())

        # free memory
        del inputs, outputs, batch_embeds
        gc.collect()
        torch.cuda.empty_cache()
    return embeddings

In [ ]:
with open('/kaggle/working/question_data.json','r',encoding='utf-8') as f:
    question_data = json.load(f)

code_embeds = {}
batch_sz = 1  # keep small for memory safety

for i in range(0, len(question_data), batch_sz):
    batch = question_data[i:i+batch_sz]
    batch_codes = []
    batch_slugs = []

    for q in batch:
        slug, details = list(q.items())[0]
        batch_codes.append(details['code'])
        batch_slugs.append(slug)

    # Encode code snippets
    batch_code_embeddings = get_embeddings(batch_codes, batch_size=1)

    for j, slug in enumerate(batch_slugs):
        code_embeds[slug] = batch_code_embeddings[j]

    # Save partial results
    with open('code_embeds.pkl', 'wb') as f:
        pickle.dump(code_embeds, f)

    print(f"Processed {i+batch_sz} / {len(question_data)}")

# Save
with open('code_embeds.pkl', 'wb') as f:
    pickle.dump(code_embeds, f)

In [ ]:
batch_sz=10 # prevent overloading the RAM
with open('/kaggle/working/question_data.json','r',encoding='utf-8') as f:
  question_data=json.load(f)

# device= 'cuda:1' if torch.cuda.is_available() else 'cpu'
# name_model.to(device)
print('loading')
for i in range(0,len(question_data),batch_sz):
    batch=question_data[i:i+batch_sz]
    batch_names=[]
    batch_slugs=[]
    for q in batch:
        slug,details=list(q.items())[0]
        batch_names.append(details['title'])
        batch_slugs.append(slug)

    batch_name_embeddings=name_model.encode(batch_names)

    for j,slug in enumerate(batch_slugs):
        name_embeds[slug]=batch_name_embeddings[j]
    with open('name_embeds.pkl', 'wb') as f:
        pickle.dump(name_embeds, f)
    print(f"Processed {i+batch_sz} questions")

In [ ]:
len(name_embeds)

In [ ]:
with open('/kaggle/working/name_embeds.pkl','rb') as f:
    data=pickle.load(f)
len(data)

# Grid Search

In [ ]:
with open('code_embeds.pkl','rb') as f:
    code_embeds=pickle.load(f)
with open('name_embeds.pkl','rb') as f:
    name_embeds=pickle.load(f)

In [ ]:
len(tags)

3624

In [ ]:
tags={}
for q in metadata:
  tags[q['titleSlug']]=q['tags']
tags

{'two-sum': ['Array', 'Hash Table'],
 'add-two-numbers': ['Linked List', 'Math', 'Recursion'],
 'longest-substring-without-repeating-characters': ['Hash Table',
  'String',
  'Sliding Window'],
 'median-of-two-sorted-arrays': ['Array',
  'Binary Search',
  'Divide and Conquer'],
 'longest-palindromic-substring': ['Two Pointers',
  'String',
  'Dynamic Programming'],
 'zigzag-conversion': ['String'],
 'reverse-integer': ['Math'],
 'string-to-integer-atoi': ['String'],
 'palindrome-number': ['Math'],
 'regular-expression-matching': ['String', 'Dynamic Programming', 'Recursion'],
 'container-with-most-water': ['Array', 'Two Pointers', 'Greedy'],
 'integer-to-roman': ['Hash Table', 'Math', 'String'],
 'roman-to-integer': ['Hash Table', 'Math', 'String'],
 'longest-common-prefix': ['Array', 'String', 'Trie'],
 '3sum': ['Array', 'Two Pointers', 'Sorting'],
 '3sum-closest': ['Array', 'Two Pointers', 'Sorting'],
 'letter-combinations-of-a-phone-number': ['Hash Table',
  'String',
  'Backtracki

In [ ]:
def baseline(weights,str_x,str_y):
  name_sim=cosine_similarity(name_embeds[str_x].reshape(1,-1),name_embeds[str_y].reshape(1,-1))[0][0]
  code_sim=cosine_similarity(code_embeds[str_x].reshape(1,-1),code_embeds[str_y].reshape(1,-1))[0][0]
  set_x, set_y = set(tags[str_x]), set(tags[str_y])
  tags_sim = len(set_x & set_y) / len(set_x | set_y) if (set_x or set_y) else 1.0
  return np.dot(weights,[name_sim,code_sim,tags_sim])

In [ ]:
param_grid={
    'w_name':np.arange(0.05,0.96,0.05),
    'w_code':np.arange(0.05,0.96,0.05),
    'w_tag':np.arange(0.05,0.41,0.05)
}

In [ ]:
grid=[p for p in ParameterGrid(param_grid) if abs(sum([p['w_name'],p['w_code'],p['w_tag']]) -1)<0.01]
best_ndcg=0
best_score=-np.inf
best_mse=np.inf
best_weight=None

In [ ]:
df=pd.read_csv('similar_questions.csv')
df.tail()

,question1,question2,score
5117,insert-into-a-sorted-circular-linked-list,frog-jump,0.0
5118,length-of-the-longest-alphabetical-continuous-...,maximum-profit-in-job-scheduling,0.0
5119,design-bounded-blocking-queue,separate-the-digits-in-an-array,0.0
5120,maximum-size-subarray-sum-equals-k,the-skyline-problem,0.0
5121,watering-plants,better-compression-of-string,0.0


In [ ]:
exclude =["last-person-to-fit-in-the-bus",'jump-game-viii'] # cleand after going through DB of questions i have
rows_to_remove_mask=(df['question1'].isin(exclude) | df['question2'].isin(exclude)) # 11
df_filtered= df[~rows_to_remove_mask]
len(df_filtered)

5111

In [ ]:
pairs=[(rows['question1'],rows['question2'],rows['score']) for _,rows in df_filtered.iterrows()]
pairs

[('two-sum', '3sum', 1.0),
 ('two-sum', '4sum', 1.0),
 ('two-sum', 'two-sum-ii-input-array-is-sorted', 1.0),
 ('two-sum', 'two-sum-iii-data-structure-design', 1.0),
 ('two-sum', 'subarray-sum-equals-k', 1.0),
 ('two-sum', 'two-sum-iv-input-is-a-bst', 1.0),
 ('two-sum', 'two-sum-less-than-k', 1.0),
 ('two-sum', 'max-number-of-k-sum-pairs', 1.0),
 ('two-sum', 'count-good-meals', 1.0),
 ('two-sum', 'count-number-of-pairs-with-absolute-difference-k', 1.0),
 ('two-sum',
  'number-of-pairs-of-strings-with-concatenation-equal-to-target',
  1.0),
 ('two-sum', 'find-all-k-distant-indices-in-an-array', 1.0),
 ('two-sum', 'first-letter-to-appear-twice', 1.0),
 ('two-sum', 'number-of-excellent-pairs', 1.0),
 ('two-sum', 'number-of-arithmetic-triplets', 1.0),
 ('two-sum', 'node-with-highest-edge-score', 1.0),
 ('two-sum', 'check-distances-between-same-letters', 1.0),
 ('two-sum', 'find-subarrays-with-equal-sum', 1.0),
 ('two-sum', 'largest-positive-integer-that-exists-with-its-negative', 1.0),
 ('t

In [ ]:
from sklearn.model_selection import train_test_split

# Split pairs (stratify to balance 1.0/0.0)
train_pairs, val_pairs = train_test_split(
    pairs,
    test_size=0.2,  # 20% (~1024 rows) for val
    stratify=[p[2] for p in pairs],  # Balance pos/neg
    random_state=42
)

# Verify balance
print("Train positives:", sum(p[2] for p in train_pairs) / len(train_pairs))
print("Val positives:", sum(p[2] for p in val_pairs) / len(val_pairs))

Train positives: 0.49902152641878667
Val positives: 0.49853372434017595


In [ ]:
collc=[]
for param in grid:
  weights=[param['w_name'],param['w_code'],param['w_tag']]
  preds=[]
  for str_x,str_y,_ in val_pairs:
    score=baseline(weights,str_x,str_y)
    preds.append(score)
  # true=[p['score'] for p in val_pairs]
  true=[p[2] for p in val_pairs]
  ndcg=ndcg_score(np.asarray([true]),np.asarray([preds]),k=10)
  mse = mean_squared_error(true, preds)
  score = ndcg - mse
  pos_preds = [preds[k] for k in range(len(true)) if true[k] == 1.0]
  neg_preds = [preds[k] for k in range(len(true)) if true[k] == 0.0]
  print(f"Weights {weights}: Avg pos score {np.mean(pos_preds) if pos_preds else 0:.4f}, Avg neg score {np.mean(neg_preds) if neg_preds else 0:.4f}, NDCG {ndcg:.4f}, MSE {mse:.4f}, Score {score:.4f}")
  if score > best_score:
      best_score = score
      best_ndcg = ndcg
      best_mse = mse
      best_weight = param
      collc = []
  elif score == best_score:
      collc.append(param)

print("\nBest Score (NDCG - MSE):", best_score)
print("Best NDCG:", best_ndcg)
print("Best MSE:", best_mse)
print("Best Weights:", best_weight)
print("Tied Weights:", collc)

In [ ]:
# Best collectiong of scores i got

# Weights [0.45, 0.15000000000000002, 0.4]: Avg pos score 0.5967, Avg neg score 0.2995, NDCG 1.0000, MSE 0.1385, Score 0.8615
# Weights [0.35000000000000003, 0.25, 0.4]: Avg pos score 0.5850, Avg neg score 0.2841, NDCG 1.0000, MSE 0.1388, Score 0.8612
# Weights [0.4, 0.25, 0.35000000000000003]: Avg pos score 0.5958, Avg neg score 0.3054, NDCG 1.0000, MSE 0.1393, Score 0.8607
# Weights [0.4, 0.2, 0.4]: Avg pos score 0.5908, Avg neg score 0.2918, NDCG 1.0000, MSE 0.1385, Score 0.8615
# Weights [0.45, 0.15000000000000002, 0.4]: Avg pos score 0.5967, Avg neg score 0.2995, NDCG 1.0000, MSE 0.1385, Score 0.8615


1. Best Score (NDCG - MSE): 0.8610283220131238
2. Best NDCG: 1.0Best MSE: 0.13897167798687615
3. Best Weights: {'w_code': 0.15000000000000002, 'w_name': 0.45, 'w_tag': 0.4}
4. Tied Weights: []


# Best combination after grid search

In [ ]:
best_weights = [0.2, 0.4, 0.4]  # [w_name, w_code, w_tag]

# Assume train_pairs = list of (str_x, str_y, temp_score); update to continuous
scored_train_pairs = []
for str_x, str_y, _ in train_pairs:
    name_sim = cosine_similarity(name_embeds[str_x].reshape(1,-1), name_embeds[str_y].reshape(1,-1))[0][0]
    code_sim = cosine_similarity(code_embeds[str_x].reshape(1,-1), code_embeds[str_y].reshape(1,-1))[0][0]
    set_x, set_y = set(tags[str_x]), set(tags[str_y])
    tags_sim = len(set_x & set_y) / len(set_x | set_y) if (set_x or set_y) else 1.0
    score = np.dot(best_weights, [name_sim, code_sim, tags_sim])
    scored_train_pairs.append((str_x, str_y, score))

# Optional: Print avg scores for sanity
pos_scores = [s[2] for s in scored_train_pairs if s[2] > 0.5]  # Approx positives
neg_scores = [s[2] for s in scored_train_pairs if s[2] <= 0.5]
print("Avg pos score:", np.mean(pos_scores) if pos_scores else 0)
print("Avg neg score:", np.mean(neg_scores) if neg_scores else 0)

Avg pos score: 0.6746705160860254
Avg neg score: 0.3008328406922815


In [ ]:
scored_val_pairs=[]
for str_x, str_y, _ in val_pairs:
    name_sim = cosine_similarity(name_embeds[str_x].reshape(1,-1), name_embeds[str_y].reshape(1,-1))[0][0]
    code_sim = cosine_similarity(code_embeds[str_x].reshape(1,-1), code_embeds[str_y].reshape(1,-1))[0][0]
    set_x, set_y = set(tags[str_x]), set(tags[str_y])
    tags_sim = len(set_x & set_y) / len(set_x | set_y) if (set_x or set_y) else 1.0
    score = np.dot(best_weights, [name_sim, code_sim, tags_sim])
    scored_val_pairs.append((str_x, str_y, score))

# Optional: Print avg scores for sanity
pos_scores = [s[2] for s in scored_val_pairs if s[2] > 0.5]  # Approx positives
neg_scores = [s[2] for s in scored_val_pairs if s[2] <= 0.5]
print("Avg pos score:", np.mean(pos_scores) if pos_scores else 0)
print("Avg neg score:", np.mean(neg_scores) if neg_scores else 0)

Avg pos score: 0.6591381800098232
Avg neg score: 0.30157238824423954


In [ ]:
from datasets import Dataset
import random

train_data = []
for str_x, str_y, score in scored_train_pairs:
    # Assume questions is dict with slug keys: questions[str_x] = {'name_text': ..., 'code_text': ..., 'tags': [...]}
    text1 = f"Name: {data[str_x]['title']} | Code: {data[str_x]['code']} | Tags: {','.join(sorted(data[str_x]['tags']))}"
    text2 = f"Name: {data[str_y]['title']} | Code: {data[str_y]['code']} | Tags: {','.join(sorted(data[str_y]['tags']))}"
    train_data.append((text1, text2, score))

In [ ]:
val_data=[]
for str_x,str_y,score in scored_val_pairs:
    text1 = f"Name: {data[str_x]['title']} | Code: {data[str_x]['code']} | Tags: {','.join(sorted(data[str_x]['tags']))}"
    text2 = f"Name: {data[str_y]['title']} | Code: {data[str_y]['code']} | Tags: {','.join(sorted(data[str_y]['tags']))}"
    val_data.append((text1, text2, score))

In [ ]:
train_data[0]

('Name: As Far from Land as Possible | Code: // Time:  O(m * n)\n// Space: O(m * n)\n\nclass Solution {\npublic:\n    int maxDistance(vector<vector<int>>& grid) {\n        static const vector<pair<int, int>> directions{{0, 1}, {1, 0}, {0, -1}, {-1, 0}};\n        queue<pair<int, int>> q;\n        for (int i = 0; i < grid.size(); ++i) {\n            for (int j = 0; j < grid[i].size(); ++j) {\n                if (grid[i][j]) {\n                    q.emplace(i, j);\n                }\n            }\n        }\n        if (q.size() == grid.size() * grid[0].size()) {\n            return -1;\n        }\n        int level = -1;\n        while (!q.empty()) {\n            queue<pair<int, int>> next_q;\n            while (!q.empty()) {\n                const auto [x, y] = q.front(); q.pop();\n                for (const auto& [dx, dy] : directions) {\n                    const auto& nx = x + dx;\n                    const auto& ny = y + dy;\n                    if (!(0 <= nx && nx < grid.size() &&

# UniXcoder fine tuning

In [ ]:
!pip uninstall -y torch-xla
!pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cpu
!pip install --upgrade sentence-transformers

Found existing installation: torch_xla 2.8.0
Uninstalling torch_xla-2.8.0:
  Successfully uninstalled torch_xla-2.8.0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.4 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.3.0
    Uninstalling torch-2.3.0:
      Successfully uninstalled torch-2.3.0
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.18.0
    Uninstalling torchvision-0.18.0:
      Successfully uninstalled torchvision-0.18.0


In [ ]:
import torch
import torch.nn as nn
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.models import Transformer, Pooling
from transformers import TrainerCallback, RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class ProgressCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"Epoch {state.epoch + 1} starting...")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Filter out None values and format nicely
            formatted_logs = {k: f"{v:.4f}" if isinstance(v, float) else v
                            for k, v in logs.items() if v is not None}
            print(f"Step {logs.get('step', 'N/A')}: {formatted_logs}")

    def on_evaluate(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Evaluation results: {logs}")

In [ ]:
def create_unixcoder_sentence_transformer(max_seq_length=512):
    """
    Create a sentence-transformer compatible UniXcoder model
    UniXcoder is based on RoBERTa architecture
    """
    print("Creating UniXcoder sentence-transformer model...")
    try:
        # Load the transformer
        word_embedding_model = Transformer(
            'microsoft/unixcoder-base',
            max_seq_length=max_seq_length,
            model_args={'trust_remote_code': True}  # might need this
        )

        # Add pooling layer
        pooling_model = Pooling(
            word_embedding_model.get_word_embedding_dimension(),
            pooling_mode_cls_token=True,    # Use [CLS] token
            pooling_mode_max_tokens=False,  # Don't use max pooling
            pooling_mode_mean_tokens=False  # Don't use mean pooling
        )

        # Create the sentence transformer
        model = SentenceTransformer(
            modules=[word_embedding_model, pooling_model]
        )

        print("UniXcoder successfully wrapped in sentence-transformers")
        return model

    except Exception as e:
        print(f"Error creating UniXcoder wrapper: {e}")
        raise e

In [ ]:
def create_training_examples(data_tuples):
    """Convert data tuples to InputExample objects for sentence-transformers"""
    examples = []
    for text1, text2, score in data_tuples:
        examples.append(InputExample(texts=[str(text1), str(text2)], label=float(score)))
    return examples

In [ ]:
def fine_tune_unixcoder(
    train_data,
    val_data=None,
    output_dir='finetuned_unixcoder_model',
    num_epochs=3,
    batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    eval_steps=100,
    save_steps=100,
    max_seq_length=512,
    loss_function='cosent'  # 'cosent', 'cosine', 'contrastive'
):
    """
    Complete UniXcoder fine-tuning pipeline
    """

    # Check device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Create UniXcoder model
    print("Loading UniXcoder model...")
    try:
        model = create_unixcoder_sentence_transformer(max_seq_length=max_seq_length)
        model = model.to(device)
        print(" UniXcoder model loaded successfully")
    except Exception as e:
        print(f" Failed to load UniXcoder: {e}")
        raise e

    train_examples = train_data
    val_examples = [
        InputExample(texts=[row["text1"], row["text2"]], label=row["score"])
        for row in val_data
    ]
    evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
        val_examples,
        name='unixcoder_leetcode_eval',
        show_progress_bar=True
    )
    print(f"Validation examples: {len(val_examples)}")
    # Loss Function
    loss = losses.CoSENTLoss(model)

    total_steps = len(train_examples) * num_epochs // batch_size
    warmup_steps = int(warmup_ratio * total_steps)

    print(f"Training configuration:")
    print(f"  Total examples: {len(train_examples)}")
    print(f"  Batch size: {batch_size}")
    print(f"  Epochs: {num_epochs}")
    print(f"  Total steps: {total_steps}")
    print(f"  Warmup steps: {warmup_steps}")
    print(f"  Learning rate: {learning_rate}")

    # Training arguments optimized for UniXcoder
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,
        warmup_steps=warmup_steps,
        learning_rate=learning_rate,

        # Mixed precision and optimization
        fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7,
        bf16=False,  # Set to True for A100/newer GPUs
        gradient_checkpointing=True,  # Save memory

        # Device settings
        use_cpu=not torch.cuda.is_available(),
        dataloader_num_workers=0,  # Avoid multiprocessing issues

        # Evaluation and saving
        eval_strategy='steps' if evaluator else 'no',
        eval_steps=eval_steps if evaluator else None,
        save_strategy='steps',
        save_steps=save_steps,
        load_best_model_at_end=True if evaluator else False,
        metric_for_best_model='eval_unixcoder_leetcode_eval_spearman_cosine' if evaluator else None,
        greater_is_better=True if evaluator else None,
        save_total_limit=3,

        # Logging
        logging_strategy='steps',
        logging_steps=50,  # More frequent logging
        report_to=None,  # Disable wandb/tensorboard

        # Reproducibility
        seed=42,
        data_seed=42,

        # Performance
        remove_unused_columns=False,
        prediction_loss_only=False,
    )

    # Create trainer
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_examples,
        loss=loss,
        evaluator=evaluator,
    )

    # Add progress callback
    trainer.add_callback(ProgressCallback())

    # Start training
    print("\n🚀 Starting UniXcoder fine-tuning...")
    print("="*60)

    try:
        trainer.train(resume_from_checkpoint=True)
        print("="*60)
        print(" Training completed successfully!")

    except Exception as e:
        print(f" Training failed: {e}")
        raise e

    # Save the model
    print(f"\n Saving model to {output_dir}")
    model.save(output_dir)

    # Create a clean final model
    final_model_path = f"{output_dir}_final"
    model.save(final_model_path)
    print(f" Final model saved to {final_model_path}")

In [ ]:
def create_training_dataset(data_tuples):
    return Dataset.from_dict({
        "text1": [str(t[0]) for t in data_tuples],
        "text2": [str(t[1]) for t in data_tuples],
        "score": [float(t[2]) for t in data_tuples],
    })

In [ ]:
train_dataset= create_training_dataset(train_data)
val_dataset = create_training_dataset(val_data)

In [ ]:
model, trainer = fine_tune_unixcoder(
      train_data=train_dataset,
      val_data=val_dataset,
      output_dir='/content/drive/MyDrive/leetcode_unixcoder',  # save path
      num_epochs=2,                        # How many times to go through the data
      batch_size=8,
      learning_rate=2e-5,
      max_seq_length=256,                 # Maximum length of input text
      loss_function='cosent'
  )

Using device: cpu
Loading UniXcoder model...
Loading UniXcoder model...
Creating UniXcoder sentence-transformer model...
UniXcoder not available directly, creating custom wrapper...


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

...

Step N/A: {'eval_unixcoder_leetcode_eval_pearson_cosine': '0.8823', 'eval_unixcoder_leetcode_eval_spearman_cosine': '0.8875', 'eval_runtime': '1662.8814', 'eval_samples_per_second': '0.0000', 'eval_steps_per_second': '0.0000', 'epoch': '1.7613'}
Step N/A: {'loss': '2.3788', 'grad_norm': '31.0104', 'learning_rate': '0.0000', 'epoch': '1.8591'}
Step N/A: {'loss': '2.3400', 'grad_norm': '31.3387', 'learning_rate': '0.0000', 'epoch': '1.9569'}


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Step N/A: {'eval_unixcoder_leetcode_eval_pearson_cosine': '0.8845', 'eval_unixcoder_leetcode_eval_spearman_cosine': '0.8840', 'eval_runtime': '1667.6961', 'eval_samples_per_second': '0.0000', 'eval_steps_per_second': '0.0000', 'epoch': '1.9569'}
Step N/A: {'train_runtime': '18447.6437', 'train_samples_per_second': '0.4430', 'train_steps_per_second': '0.0550', 'total_flos': '0.0000', 'train_loss': '0.5353', 'epoch': '2.0000'}
✅ Training completed successfully!

💾 Saving model to /content/drive/MyDrive/leetcode_unixcoder
✅ Final model saved to /content/drive/MyDrive/leetcode_unixcoder_final


TypeError: cannot unpack non-iterable NoneType object